In [6]:
# import & setting 
import scanpy as sc
import anndata as ad
import pandas as pd 
import matplotlib.pyplot as plt
from pathlib import Path

# configure plotting parameters
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 120, facecolor= "white", frameon = False )

In [11]:
# load the all 8 data samples into single unified anndata object
!ls

LICENSE    Untitled.ipynb  notebooks	       report	scripts
README.md  data		   project1_guide.pdf  results	workflow.py


In [19]:
from pathlib import Path
import scanpy as sc

data_dir = Path("data")

# Locate all sample folders containing 10x matrices
sample_dirs = sorted([
    p for p in data_dir.iterdir()
    if p.is_dir() and (p / "filtered_feature_bc_matrix").exists()
])

print(f"Discovered {len(sample_dirs)} samples: {[p.name for p in sample_dirs]}")

# Load files into dictionary and assign metadata annotations
adatas = {}

for sample_path in sample_dirs:
    sample_name = sample_path.name
    matrix_dir = sample_path / "filtered_feature_bc_matrix"

    # Read 10x matrix directory
    adata_sample = sc.read_10x_mtx(
        matrix_dir,
        var_names="gene_symbols",
        cache=True
    )
    
    adata_sample.var_names_make_unique()
    adata_sample.obs_names_make_unique()

    # Assign metadata annotations
    adata_sample.obs["sample"] = sample_name

    # Parse condition (ctrl, 3dp, 7dp, 10dp) and replicate (1, 2)
    if sample_name.startswith("ctrl"):
        adata_sample.obs["condition"] = "ctrl"
        adata_sample.obs["replicate"] = sample_name.replace("ctrl", "")
    elif "dp" in sample_name:
        cond, rep = sample_name.split("dp")
        adata_sample.obs["condition"] = f"{cond}dp"
        adata_sample.obs["replicate"] = rep
    else:
        adata_sample.obs["condition"] = sample_name
        adata_sample.obs["replicate"] = "1"

    # Store in dictionary after adding metadata
    adatas[sample_name] = adata_sample
    
# Merge all samples into one AnnData object
adata = ad.concat(adatas, label="sample_batch", index_unique="-", join="outer")
adata.var_names_make_unique()
print(f"Total dataset dimensions: {adata.n_obs} cells x {adata.n_vars} genes")


Discovered 8 samples: ['10dp1', '10dp2', '3dp1', '3dp2', '7dp1', '7dp2', 'ctrl1', 'ctrl2']
... reading from cache file cache/data-10dp1-filtered_feature_bc_matrix-matrix.h5ad
... reading from cache file cache/data-10dp2-filtered_feature_bc_matrix-matrix.h5ad
... reading from cache file cache/data-3dp1-filtered_feature_bc_matrix-matrix.h5ad
... reading from cache file cache/data-3dp2-filtered_feature_bc_matrix-matrix.h5ad
... reading from cache file cache/data-7dp1-filtered_feature_bc_matrix-matrix.h5ad
... reading from cache file cache/data-7dp2-filtered_feature_bc_matrix-matrix.h5ad
... reading from cache file cache/data-ctrl1-filtered_feature_bc_matrix-matrix.h5ad
... reading from cache file cache/data-ctrl2-filtered_feature_bc_matrix-matrix.h5ad
Total dataset dimensions: 20097 cells x 25433 genes


In [24]:
adata.obs

,sample,condition,replicate,sample_batch
AAACCCAAGATTAGTG-1-10dp1,10dp1,10dp,1,10dp1
AAACCCAGTCTACGAT-1-10dp1,10dp1,10dp,1,10dp1
AAACCCAGTGTTAGCT-1-10dp1,10dp1,10dp,1,10dp1
AAACGAAAGGGCGAAG-1-10dp1,10dp1,10dp,1,10dp1
AAACGAAAGGTCATAA-1-10dp1,10dp1,10dp,1,10dp1
...,...,...,...,...
TTTGGAGCATTAGGCT-1-ctrl2,ctrl2,ctrl,2,ctrl2
TTTGGAGGTGAATTGA-1-ctrl2,ctrl2,ctrl,2,ctrl2
TTTGGTTCAACAGCCC-1-ctrl2,ctrl2,ctrl,2,ctrl2
TTTGGTTCACGCGCAT-1-ctrl2,ctrl2,ctrl,2,ctrl2


In [25]:
adata.var_names

Index(['fgfr1op2', 'si:dkey-21h14.12', 'si:dkey-285e18.2', 'znf1114',
       'si:dkey-21h14.10', 'ERC1', 'si:dkey-199m13.5', 'erc1b',
       'si:dkey-199m13.4', 'si:dkey-285e18.5',
       ...
       'ttc8', 'CABZ01074745.1', 'CABZ01118270.1', 'PCNP', 'CABZ01088864.1',
       'CABZ01110379.1', 'WARS1', 'LAMP5', 'RHO', 'EGFP'],
      dtype='object', length=25433)

In [26]:
adata.shape

(20097, 25433)